# RAG Pipeline using Gemini + Pinecone (.env Version)

In [14]:
%pip install -q langchain langchain-community langchain-text-splitters pypdf google-generativeai pinecone python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import os
import re
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv

import google.generativeai as genai
from pinecone import Pinecone, ServerlessSpec
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_HOST = os.getenv("PINECONE_HOST")

class LocalInMemoryIndex:
    def __init__(self):
        self.documents = []

    def upsert(self, vectors):
        for vector_id, embedding, metadata in vectors:
            self.documents.append((vector_id, embedding, metadata))

    def query(self, vector, top_k=5, include_metadata=True):
        def similarity(a, b):
            if isinstance(a, dict) and isinstance(b, dict):
                overlap = set(a.keys()) & set(b.keys())
                return sum(a[token] * b[token] for token in overlap)
            if isinstance(a, list) and isinstance(b, list):
                return sum(x * y for x, y in zip(a, b))
            return 0

        scored = []
        for vector_id, embedding, metadata in self.documents:
            scored.append((similarity(vector, embedding), vector_id, metadata))

        scored.sort(key=lambda item: item[0], reverse=True)
        matches = []
        for score, vector_id, metadata in scored[:top_k]:
            if include_metadata:
                matches.append({"id": vector_id, "score": score, "metadata": metadata})
            else:
                matches.append({"id": vector_id, "score": score})
        return {"matches": matches}


def simple_embedding(text):
    tokens = re.findall(r"[a-zA-Z0-9]+", text.lower())
    counts = Counter(tokens)
    return dict(counts)


def get_embedding(text):
    if GEMINI_API_KEY:
        try:
            return genai.embed_content(
                model="models/gemini-embedding-001",
                content=text,
                task_type="retrieval_document"
            )["embedding"]
        except Exception as exc:
            print(f"Gemini embedding failed, using local fallback: {exc}")
    return simple_embedding(text)

pc = None
index = None

if GEMINI_API_KEY:
    try:
        genai.configure(api_key=GEMINI_API_KEY)
    except Exception as exc:
        print(f"Gemini configuration failed: {exc}")

if PINECONE_API_KEY and not PINECONE_API_KEY.startswith("https://"):
    try:
        if PINECONE_HOST:
            pc = Pinecone(api_key=PINECONE_API_KEY, host=PINECONE_HOST)
        else:
            pc = Pinecone(api_key=PINECONE_API_KEY)

        index_name = "rag-notes"
        if index_name not in [i["name"] for i in pc.list_indexes()]:
            pc.create_index(
                name=index_name,
                dimension=768,
                metric="cosine",
                spec=ServerlessSpec(cloud="aws", region="us-east-1")
            )

        index = pc.Index(index_name)
        print("Using Pinecone index")
    except Exception as exc:
        print(f"Pinecone unavailable, using local fallback: {exc}")

if index is None:
    index = LocalInMemoryIndex()
    print("Using local in-memory index")


Using local in-memory index


In [16]:
docs = []

source_dir = Path("data")
if source_dir.exists():
    for file in source_dir.glob("*"):
        if file.suffix.lower() == ".pdf":
            docs.extend(PyPDFLoader(str(file)).load())
        elif file.suffix.lower() == ".txt":
            docs.extend(TextLoader(str(file), encoding="utf-8").load())
else:
    print("No data folder found. Create a 'data' folder and add PDF/TXT files.")

print("Loaded:", len(docs), "documents")


Loaded: 153 documents


In [17]:

splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=120
)

chunks = splitter.split_documents(docs)
print("Chunks:", len(chunks))


Chunks: 303


In [18]:

def get_embedding(text):
    return genai.embed_content(
        model="models/gemini-embedding-001",
        content=text,
        task_type="retrieval_document"
    )["embedding"]


In [19]:

vectors = []

for i, chunk in enumerate(chunks):
    vectors.append((
        str(i),
        get_embedding(chunk.page_content),
        {
            "text": chunk.page_content,
            "source": chunk.metadata.get("source", "")
        }
    ))

index.upsert(vectors=vectors)
print("Vectors uploaded")


Vectors uploaded


In [20]:

def search_docs(question, top_k=5):
    query_vector = genai.embed_content(
        model="models/gemini-embedding-001",
        content=question,
        task_type="retrieval_query"
    )["embedding"]

    return index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )["matches"]


In [23]:

model = genai.GenerativeModel("gemini-2.5-flash")

def ask(question):
    matches = search_docs(question)

    context = ""

    for item in matches:
        context += item["metadata"]["text"] + "\n\n"

    prompt = f"""Use only the context below to answer.

Context:
{context}

Question:
{question}
"""

    response = model.generate_content(prompt)

    print(response.text)

    print("\nSources:")
    for item in matches:
        print("-", item["metadata"]["source"])

ask("What is stack in depth?")


A stack is a linear data structure where insertion and deletion of items takes place at one end called the top of the stack. It operates on a last-in first-out (LIFO) basis.

Key characteristics and terminology:
*   **Mechanism:** It uses a single index or pointer to keep track of the information.
*   **Basic Operations:**
    *   `push` (insert): Placing an item onto the stack.
    *   `pop` (remove): Removing an item from the stack.
*   **Stack Pointer:** Keeps track of the current position on the stack. When an element is placed, it's "pushed"; when removed, it's "popped."
*   **Error Conditions:**
    *   **Overflow:** Occurs when attempting to push more information onto a stack than it can hold.
    *   **Underflow:** Occurs when attempting to pop an item off an empty stack.
*   **Implementation Hint:** Assume array elements begin at 0, a maximum number of elements (`max`) can be placed, and the stack pointer is referred to as `top`.

Sources:
- data\dsa.pdf
- data\dsa.pdf
- data\